In [38]:
import numpy as np

np.random.seed(42)

print("=" * 80)
print("TASK 3: APPLIED DATA ANALYSIS - FITNESS TRACKING")
print("=" * 80)


In [39]:
print("\n### PART A: DATA GENERATION & PREPARATION ###\n")
print("-" * 80)

num_users = 100
num_days = 90
num_metrics = 4

print("Generating fitness tracking data...")
print(f"Dimensions: {num_users} users × {num_days} days × {num_metrics} metrics")


Dimensions: 100 users × 90 days × 4 metrics


In [40]:
fitness_data = np.zeros((num_users, num_days, num_metrics))

fitness_data[:, :, 0] = np.random.normal(loc=8500, scale=2500, size=(num_users, num_days))
fitness_data[:, :, 0] = np.clip(fitness_data[:, :, 0], 2000, 15000)

fitness_data[:, :, 1] = np.random.normal(loc=2500, scale=400, size=(num_users, num_days))
fitness_data[:, :, 1] = np.clip(fitness_data[:, :, 1], 1500, 3500)

fitness_data[:, :, 2] = np.random.normal(loc=100, scale=30, size=(num_users, num_days))
fitness_data[:, :, 2] = np.clip(fitness_data[:, :, 2], 20, 180)

fitness_data[:, :, 3] = np.random.normal(loc=85, scale=12, size=(num_users, num_days))
fitness_data[:, :, 3] = np.clip(fitness_data[:, :, 3], 60, 120)

print(f"Data shape: {fitness_data.shape}")
print(f"Sample data (User 1, Day 1): {fitness_data[0, 0, :]}")


Data shape: (100, 90, 4)
Sample data (User 1, Day 1): [9741.78538253 2806.16086241  109.6550557    75.59487797]


In [41]:
print("\nGenerating user metadata...")
user_metadata = np.zeros((num_users, 3))
user_metadata[:, 0] = np.arange(1, num_users + 1)
user_metadata[:, 1] = np.random.randint(18, 71, size=num_users)
user_metadata[:, 2] = np.random.randint(0, 2, size=num_users)

print(f"Metadata shape: {user_metadata.shape}")
print(f"Sample metadata (first 5 users):")
print("User ID | Age | Gender")
for i in range(5):
    gender_label = "Male" if user_metadata[i, 2] == 1 else "Female"
    print(f"  {int(user_metadata[i, 0]):3d}   | {int(user_metadata[i, 1]):2d}  | {gender_label}")


Metadata shape: (100, 3)
Sample metadata (first 5 users):
User ID | Age | Gender
    1   | 43  | Male
    2   | 70  | Female
    3   | 46  | Female
    4   | 35  | Female
    5   | 26  | Female


In [42]:
print("\nIntroducing data quality issues...")

total_elements = fitness_data.size
nan_count = int(total_elements * 0.05)
nan_indices = np.random.choice(total_elements, nan_count, replace=False)
flat_data = fitness_data.flatten()
flat_data[nan_indices] = np.nan
fitness_data = flat_data.reshape(fitness_data.shape)

print(f"Inserted {nan_count} NaN values ({(nan_count/total_elements)*100:.1f}% of data)")


Inserted 1800 NaN values (5.0% of data)


In [43]:
outlier_count = int(total_elements * 0.02)
outlier_indices = np.random.choice(total_elements, outlier_count, replace=False)

flat_data = fitness_data.flatten()
for idx in outlier_indices:
    metric_type = idx % num_metrics
    if metric_type == 0:
        flat_data[idx] = np.random.choice([500, 50000])
    elif metric_type == 1:
        flat_data[idx] = np.random.choice([500, 8000])
    elif metric_type == 2:
        flat_data[idx] = np.random.choice([5, 400])
    else:
        flat_data[idx] = np.random.choice([30, 180])

fitness_data = flat_data.reshape(fitness_data.shape)
print(f"Inserted {outlier_count} outlier values ({(outlier_count/total_elements)*100:.1f}% of data)")

print(f"\nData preparation complete!")
print(f"Total data quality issues: {nan_count + outlier_count} ({((nan_count + outlier_count)/total_elements)*100:.1f}%)")


Inserted 720 outlier values (2.0% of data)
Total data quality issues: 2520 (7.0%)


In [44]:
print("\n\n### PART B: DATA CLEANING & VALIDATION ###\n")
print("-" * 80)

def handle_missing(data):
    data_copy = data.copy()

    for metric in range(data_copy.shape[2]):
        metric_data = data_copy[:, :, metric]
        metric_mean = np.nanmean(metric_data)
        metric_data[np.isnan(metric_data)] = metric_mean
        data_copy[:, :, metric] = metric_data

    return data_copy

def remove_outliers(data, metric_index):
    data_copy = data.copy()
    metric_data = data_copy[:, :, metric_index].flatten()

    q1 = np.percentile(metric_data[~np.isnan(metric_data)], 25)
    q3 = np.percentile(metric_data[~np.isnan(metric_data)], 75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (metric_data < lower_bound) | (metric_data > upper_bound)
    outlier_count = np.sum(outlier_mask & ~np.isnan(metric_data))

    median_value = np.nanmedian(metric_data)
    metric_data[outlier_mask] = median_value

    data_copy[:, :, metric_index] = metric_data.reshape(data_copy.shape[0], data_copy.shape[1])

    return data_copy, outlier_count, lower_bound, upper_bound


In [45]:
print("Starting data cleaning pipeline...\n")

fitness_data_original = fitness_data.copy()

initial_nan_count = np.sum(np.isnan(fitness_data))
print(f"Initial NaN count: {initial_nan_count}")

metric_names = ['Daily Steps', 'Calories', 'Active Minutes', 'Avg Heart Rate']

for metric_idx in range(num_metrics):
    fitness_data, outliers_removed, lower, upper = remove_outliers(fitness_data, metric_idx)
    print(f"\n{metric_names[metric_idx]}:")
    print(f"  Outliers removed: {outliers_removed}")
    print(f"  Valid range: [{lower:.2f}, {upper:.2f}]")


Starting data cleaning pipeline...

Initial NaN count: 1772

Daily Steps:
  Outliers removed: 182
  Valid range: [1617.32, 15402.25]

Calories:
  Outliers removed: 195
  Valid range: [1400.71, 3605.00]

Active Minutes:
  Outliers removed: 171
  Valid range: [17.14, 182.74]

Avg Heart Rate:
  Outliers removed: 194
  Valid range: [50.42, 119.02]


In [46]:
print("\n" + "-" * 80)
print("Handling missing values...")
fitness_data = handle_missing(fitness_data)

final_nan_count = np.sum(np.isnan(fitness_data))
print(f"\nFinal NaN count: {final_nan_count}")

if final_nan_count == 0:
    print("✓ Data cleaning successful! No NaN values remain.")
else:
    print(f"⚠ Warning: {final_nan_count} NaN values still present.")

print(f"\nCleaned data statistics:")
for metric_idx, name in enumerate(metric_names):
    metric_data = fitness_data[:, :, metric_idx]
    print(f"{name}: Mean={metric_data.mean():.2f}, Std={metric_data.std():.2f}, "
          f"Min={metric_data.min():.2f}, Max={metric_data.max():.2f}")



Final NaN count: 0
Data cleaning successful! No NaN values remain.

Cleaned data statistics:
Daily Steps: Mean=8509.73, Std=2407.01, Min=2000.00, Max=15000.00
Calories: Mean=2503.59, Std=379.68, Min=1500.00, Max=3500.00
Active Minutes: Mean=100.11, Std=28.39, Min=20.00, Max=180.00
Avg Heart Rate: Mean=84.78, Std=11.41, Min=60.00, Max=119.00


In [47]:
print("\n\n### PART C: COMPREHENSIVE ANALYSIS ###\n")
print("=" * 80)

print("\n1. USER BEHAVIOR PATTERNS")
print("-" * 80)

user_avg_metrics = fitness_data.mean(axis=1)

print("Average metrics per user (showing first 5 users):")
print("User | Steps   | Calories | Active Min | Heart Rate")
for i in range(5):
    print(f"  {int(user_metadata[i, 0]):2d} | {user_avg_metrics[i, 0]:7.1f} | "
          f"{user_avg_metrics[i, 1]:8.1f} | {user_avg_metrics[i, 2]:10.1f} | "
          f"{user_avg_metrics[i, 3]:10.1f}")


Average metrics per user (showing first 5 users):
User | Steps   | Calories | Active Min | Heart Rate
   1 |  8295.2 |   2496.4 |      100.0 |       83.5
   2 |  8605.8 |   2559.6 |      106.1 |       83.5
   3 |  8540.4 |   2432.3 |       99.8 |       84.0
   4 |  8559.7 |   2517.5 |      102.6 |       84.9
   5 |  8682.2 |   2473.5 |       99.5 |       85.1


In [48]:
print("\nIdentifying top 10 most active users...")

z_scores = np.zeros_like(user_avg_metrics)
for metric in range(num_metrics):
    mean = user_avg_metrics[:, metric].mean()
    std = user_avg_metrics[:, metric].std()
    z_scores[:, metric] = (user_avg_metrics[:, metric] - mean) / std

combined_z_score = z_scores.sum(axis=1)
top_10_indices = np.argsort(combined_z_score)[-10:][::-1]

print("\nTop 10 Most Active Users:")
print("Rank | User ID | Combined Z-Score | Steps   | Calories")
for rank, idx in enumerate(top_10_indices, 1):
    user_id = int(user_metadata[idx, 0])
    print(f" {rank:2d}  |   {user_id:3d}   | {combined_z_score[idx]:15.2f} | "
          f"{user_avg_metrics[idx, 0]:7.1f} | {user_avg_metrics[idx, 1]:8.1f}")



Top 10 Most Active Users:
Rank | User ID | Combined Z-Score | Steps   | Calories
  1  |    67   |            4.73 |  8657.3 |   2553.1
  2  |    30   |            4.51 |  8742.8 |   2538.2
  3  |    95   |            3.75 |  8604.2 |   2532.6
  4  |    84   |            3.73 |  8558.0 |   2560.7
  5  |    78   |            3.45 |  8903.2 |   2558.0
  6  |    10   |            3.12 |  9131.0 |   2495.9
  7  |    61   |            3.06 |  8390.4 |   2503.5
  8  |    97   |            3.04 |  8908.6 |   2564.4
  9  |    35   |            2.82 |  8574.2 |   2524.5
 10  |    17   |            2.70 |  9058.1 |   2486.8


In [49]:
print("\nFinding most consistent users...")
user_std_steps = fitness_data[:, :, 0].std(axis=1)
most_consistent_indices = np.argsort(user_std_steps)[:10]

print("\nTop 10 Most Consistent Users (by steps):")
print("Rank | User ID | Std Dev | Avg Steps")
for rank, idx in enumerate(most_consistent_indices, 1):
    user_id = int(user_metadata[idx, 0])
    print(f" {rank:2d}  |   {user_id:3d}   | {user_std_steps[idx]:7.2f} | {user_avg_metrics[idx, 0]:9.1f}")



Top 10 Most Consistent Users (by steps):
Rank | User ID | Std Dev | Avg Steps
  1  |     4   | 1883.28 |    8559.7
  2  |    53   | 2004.78 |    8521.6
  3  |    43   | 2023.56 |    8046.7
  4  |    41   | 2043.18 |    8560.7
  5  |    30   | 2073.30 |    8742.8
  6  |   100   | 2101.64 |    8720.7
  7  |    51   | 2103.42 |    8591.5
  8  |    98   | 2115.21 |    8577.4
  9  |    61   | 2127.85 |    8390.4
 10  |    23   | 2134.02 |    8299.9


In [50]:
print("\nClassifying users by activity level...")
steps_25th = np.percentile(user_avg_metrics[:, 0], 25)
steps_75th = np.percentile(user_avg_metrics[:, 0], 75)

low_activity = np.sum(user_avg_metrics[:, 0] < steps_25th)
medium_activity = np.sum((user_avg_metrics[:, 0] >= steps_25th) &
                         (user_avg_metrics[:, 0] <= steps_75th))
high_activity = np.sum(user_avg_metrics[:, 0] > steps_75th)

print(f"\nActivity Level Distribution:")
print(f"  Low Activity (< {steps_25th:.0f} steps): {low_activity} users ({low_activity}%)")
print(f"  Medium Activity ({steps_25th:.0f}-{steps_75th:.0f} steps): {medium_activity} users ({medium_activity}%)")
print(f"  High Activity (> {steps_75th:.0f} steps): {high_activity} users ({high_activity}%)")



Activity Level Distribution:
  Low Activity (< 8354 steps): 25 users (25%)
  Medium Activity (8354-8659 steps): 50 users (50%)
  High Activity (> 8659 steps): 25 users (25%)


In [51]:
print("\n\n2. TEMPORAL TRENDS")
print("-" * 80)

print("Computing 7-day rolling averages...")

population_daily_avg = fitness_data.mean(axis=0)

rolling_avg_7day = np.zeros((num_days - 6, num_metrics))
for metric in range(num_metrics):
    rolling_avg_7day[:, metric] = np.convolve(population_daily_avg[:, metric],
                                               np.ones(7)/7, mode='valid')

print(f"7-day rolling average shape: {rolling_avg_7day.shape}")
print(f"First 5 days (Steps): {rolling_avg_7day[:5, 0]}")


7-day rolling average shape: (84, 4)
First 5 days (Steps): [8317.92518963 8339.29737667 8410.51790098 8369.03087186 8444.34585756]


In [52]:
print("\nAnalyzing weekly patterns...")
day_of_week_avg = np.zeros((7, num_metrics))

for day_idx in range(7):
    days_in_category = population_daily_avg[day_idx::7, :]
    day_of_week_avg[day_idx, :] = days_in_category.mean(axis=0)

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print("\nAverage Activity by Day of Week:")
print("Day    | Steps   | Calories | Active Min | Heart Rate")
for day_idx, day_name in enumerate(day_names):
    print(f"{day_name}    | {day_of_week_avg[day_idx, 0]:7.1f} | "
          f"{day_of_week_avg[day_idx, 1]:8.1f} | {day_of_week_avg[day_idx, 2]:10.1f} | "
          f"{day_of_week_avg[day_idx, 3]:10.1f}")



Average Activity by Day of Week:
Day    | Steps   | Calories | Active Min | Heart Rate
Mon    |  8528.3 |   2508.6 |      100.7 |       84.5
Tue    |  8603.8 |   2490.2 |       99.5 |       84.8
Wed    |  8503.0 |   2494.8 |       99.2 |       84.6
Thu    |  8554.8 |   2507.6 |       99.2 |       84.7
Fri    |  8479.3 |   2524.7 |      100.7 |       84.8
Sat    |  8429.6 |   2500.9 |      100.5 |       85.4
Sun    |  8466.0 |   2497.8 |      101.1 |       84.4


In [53]:
print("\nAnalyzing activity trends over time...")

first_30_days = population_daily_avg[:30, :].mean(axis=0)
last_30_days = population_daily_avg[-30:, :].mean(axis=0)

print("\nFirst 30 Days vs Last 30 Days:")
for metric_idx, name in enumerate(metric_names):
    change = last_30_days[metric_idx] - first_30_days[metric_idx]
    pct_change = (change / first_30_days[metric_idx]) * 100
    trend = "↑" if change > 0 else "↓"
    print(f"{name}: {first_30_days[metric_idx]:.1f} → {last_30_days[metric_idx]:.1f} "
          f"({trend} {abs(pct_change):.1f}%)")



First 30 Days vs Last 30 Days:
Daily Steps: 8506.9 → 8530.6 (↑ 0.3%)
Calories: 2508.2 → 2503.7 (↓ 0.2%)
Active Minutes: 101.1 → 98.7 (↓ 2.3%)
Avg Heart Rate: 84.7 → 84.9 (↑ 0.3%)


In [54]:
print("\nMonth-over-Month Growth Rates:")
month1_avg = population_daily_avg[:30, :].mean(axis=0)
month2_avg = population_daily_avg[30:60, :].mean(axis=0)
month3_avg = population_daily_avg[60:90, :].mean(axis=0)

growth_m1_m2 = ((month2_avg - month1_avg) / month1_avg) * 100
growth_m2_m3 = ((month3_avg - month2_avg) / month2_avg) * 100

print("\nMonth 1 → Month 2:")
for metric_idx, name in enumerate(metric_names):
    print(f"  {name}: {growth_m1_m2[metric_idx]:+.2f}%")

print("\nMonth 2 → Month 3:")
for metric_idx, name in enumerate(metric_names):
    print(f"  {name}: {growth_m2_m3[metric_idx]:+.2f}%")



Month 1 → Month 2:
  Daily Steps: -0.18%
  Calories: -0.37%
  Active Minutes: -0.50%
  Avg Heart Rate: +0.10%

Month 2 → Month 3:
  Daily Steps: +0.46%
  Calories: +0.19%
  Active Minutes: -1.82%
  Avg Heart Rate: +0.19%


In [55]:
print("\n\n3. CORRELATIONS & INSIGHTS")
print("-" * 80)

print("Computing correlation matrix...")
metrics_flat = fitness_data.reshape(-1, num_metrics)
correlation_matrix = np.corrcoef(metrics_flat.T)

print("\nCorrelation Matrix (Metrics):")
print("           Steps  Calories  Active  Heart Rate")
for idx, name in enumerate(metric_names):
    print(f"{name:12s} ", end="")
    for corr_val in correlation_matrix[idx, :]:
        print(f"{corr_val:7.3f}  ", end="")
    print()



Correlation Matrix (Metrics):
           Steps  Calories  Active  Heart Rate
Daily Steps    1.000    0.001    0.002   -0.014  
Calories       0.001    1.000    0.014    0.008  
Active Minutes   0.002    0.014    1.000   -0.002  
Avg Heart Rate  -0.014    0.008   -0.002    1.000  


In [56]:
print("\nAnalyzing age vs activity relationship...")
age_groups = [(18, 30), (31, 45), (46, 60), (61, 70)]
age_activity = np.zeros((len(age_groups), num_metrics))

for idx, (min_age, max_age) in enumerate(age_groups):
    age_mask = (user_metadata[:, 1] >= min_age) & (user_metadata[:, 1] <= max_age)
    if np.sum(age_mask) > 0:
        age_activity[idx, :] = user_avg_metrics[age_mask, :].mean(axis=0)

print("\nAverage Activity by Age Group:")
print("Age Group  | Steps   | Calories | Active Min")
for idx, (min_age, max_age) in enumerate(age_groups):
    print(f"{min_age}-{max_age}      | {age_activity[idx, 0]:7.1f} | "
          f"{age_activity[idx, 1]:8.1f} | {age_activity[idx, 2]:10.1f}")



Average Activity by Age Group:
Age Group  | Steps   | Calories | Active Min
18-30      |  8542.2 |   2498.9 |      100.0
31-45      |  8533.5 |   2504.6 |      100.2
46-60      |  8480.5 |   2497.1 |      100.0
61-70      |  8463.0 |   2516.1 |      100.2


In [57]:
print("\nGender-based activity comparison...")
female_mask = user_metadata[:, 2] == 0
male_mask = user_metadata[:, 2] == 1

female_avg = user_avg_metrics[female_mask, :].mean(axis=0)
male_avg = user_avg_metrics[male_mask, :].mean(axis=0)

print("\nAverage Activity by Gender:")
print("Gender | Steps   | Calories | Active Min | Heart Rate")
print(f"Female | {female_avg[0]:7.1f} | {female_avg[1]:8.1f} | "
      f"{female_avg[2]:10.1f} | {female_avg[3]:10.1f}")
print(f"Male   | {male_avg[0]:7.1f} | {male_avg[1]:8.1f} | "
      f"{male_avg[2]:10.1f} | {male_avg[3]:10.1f}")



Average Activity by Gender:
Gender | Steps   | Calories | Active Min | Heart Rate
Female |  8490.8 |   2503.9 |      100.0 |       84.7
Male   |  8527.9 |   2503.3 |      100.3 |       84.8


In [58]:
print("\nCalculating Health Score...")
steps_normalized = (user_avg_metrics[:, 0] - user_avg_metrics[:, 0].min()) / \
                   (user_avg_metrics[:, 0].max() - user_avg_metrics[:, 0].min()) * 100
calories_normalized = (user_avg_metrics[:, 1] - user_avg_metrics[:, 1].min()) / \
                      (user_avg_metrics[:, 1].max() - user_avg_metrics[:, 1].min()) * 100
active_normalized = (user_avg_metrics[:, 2] - user_avg_metrics[:, 2].min()) / \
                    (user_avg_metrics[:, 2].max() - user_avg_metrics[:, 2].min()) * 100

health_score = (steps_normalized * 0.4 + calories_normalized * 0.3 + active_normalized * 0.3)

top_5_health = np.argsort(health_score)[-5:][::-1]
print("\nTop 5 Users by Health Score:")
print("Rank | User ID | Health Score")
for rank, idx in enumerate(top_5_health, 1):
    user_id = int(user_metadata[idx, 0])
    print(f" {rank}   |   {user_id:3d}   | {health_score[idx]:12.2f}")



Top 5 Users by Health Score:
Rank | User ID | Health Score
 1   |    78   |        82.80
 2   |    93   |        82.60
 3   |    67   |        79.87
 4   |     2   |        79.15
 5   |    97   |        77.74


In [59]:
print("\n\n4. GOAL ACHIEVEMENT")
print("-" * 80)

GOAL_STEPS = 8000
GOAL_CALORIES = 2000
GOAL_ACTIVE_MIN = 60

print(f"Daily Goals: {GOAL_STEPS} steps, {GOAL_CALORIES} calories, {GOAL_ACTIVE_MIN} active minutes")

steps_goal_met = (fitness_data[:, :, 0] >= GOAL_STEPS).sum(axis=1) / num_days * 100
calories_goal_met = (fitness_data[:, :, 1] >= GOAL_CALORIES).sum(axis=1) / num_days * 100
active_goal_met = (fitness_data[:, :, 2] >= GOAL_ACTIVE_MIN).sum(axis=1) / num_days * 100

all_goals_met = ((fitness_data[:, :, 0] >= GOAL_STEPS) &
                 (fitness_data[:, :, 1] >= GOAL_CALORIES) &
                 (fitness_data[:, :, 2] >= GOAL_ACTIVE_MIN)).sum(axis=1) / num_days * 100

print("\nGoal Achievement Summary:")
print(f"Average Steps Goal Achievement: {steps_goal_met.mean():.1f}%")
print(f"Average Calories Goal Achievement: {calories_goal_met.mean():.1f}%")
print(f"Average Active Minutes Goal Achievement: {active_goal_met.mean():.1f}%")
print(f"Average All Goals Met: {all_goals_met.mean():.1f}%")


Daily Goals: 8000 steps, 2000 calories, 60 active minutes

Goal Achievement Summary:
Average Steps Goal Achievement: 61.0%
Average Calories Goal Achievement: 90.5%
Average Active Minutes Goal Achievement: 92.0%
Average All Goals Met: 51.1%


In [60]:
consistent_achievers = np.where(all_goals_met > 80)[0]
print(f"\nUsers who meet ALL goals >80% of the time: {len(consistent_achievers)} users")

if len(consistent_achievers) > 0:
    print("\nTop Consistent Goal Achievers:")
    print("User ID | Achievement Rate | Avg Steps | Avg Calories | Avg Active Min")
    for idx in consistent_achievers[:10]:
        user_id = int(user_metadata[idx, 0])
        print(f"  {user_id:3d}   | {all_goals_met[idx]:15.1f}% | "
              f"{user_avg_metrics[idx, 0]:9.1f} | {user_avg_metrics[idx, 1]:12.1f} | "
              f"{user_avg_metrics[idx, 2]:14.1f}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)



Users who meet ALL goals >80% of the time: 0 users
